<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_HALLUCINATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers huggingface_hub unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 

In [1]:
!pip show transformers huggingface_hub unsloth

Name: transformers
Version: 5.5.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers, trl, unsloth, unsloth_zoo
---
Name: huggingface_hub
Version: 1.28.0
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache-2.0
Locat

In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
import contextlib
import io

print("="*80)
print("🧪 TOPO-2026 FULL EVALUATION & DATA-DRIVEN HALLUCINATION AUDIT")
print("   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026")
print("="*80)

# 1. Configuration
REPO_ID = "frankmorales2020/gemma-4-e4b-stl10-topo-2026"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 64

# 2. Load Base Model with Unsloth / Transformers Fallback
print("\n👁️ Loading Vision Model...")
vision_model = None
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel
        vision_model, vision_processor = FastVisionModel.from_pretrained(
            "frankmorales2020/gemma-4-e4b-unesco-optimized",
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(vision_model)
    print("✅ Gemma Loaded (Unsloth)")
except Exception as e:
    print(f"⚠️ Unsloth failed: {e}")
    from transformers import AutoModelForCausalLM
    vision_model = AutoModelForCausalLM.from_pretrained(
        "frankmorales2020/gemma-4-e4b-unesco-optimized",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print("✅ Gemma Loaded (Transformers)")

vision_model = vision_model.to(DEVICE)
for param in vision_model.parameters():
    param.requires_grad = False

# 3. Download Checkpoint from HF
print("\n📥 Downloading trained weights from Hugging Face...")
ckpt_path = hf_hub_download(REPO_ID, "topo_trained_parts_gemma_5runs.pt")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"   ✅ Checkpoint loaded! Best Task C Accuracy: {ckpt['best_acc_c']*100:.2f}%")

# 4. Load Tokenizer from HF
print("\n📥 Loading tokenizer from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 5. Build Classifier Model & Load Weights
print("\n🏗️ Building classifier model...")
class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1].float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

model = GemmaTopoClassifier(vision_model, ckpt['hidden_size']).to(DEVICE)
model.classifier_A.load_state_dict(ckpt["classifier_A"])
model.classifier_B.load_state_dict(ckpt["classifier_B"])
model.classifier_C.load_state_dict(ckpt["classifier_C"])

with torch.no_grad():
    emb_weight = ckpt["embed_tokens_weight"].to(DEVICE)
    embed_layer = vision_model.get_input_embeddings()
    if emb_weight.shape != embed_layer.weight.shape:
        if emb_weight.shape[0] < embed_layer.weight.shape[0]:
            pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
            pad = torch.randn(pad_size, emb_weight.shape[1], device=DEVICE)
            emb_weight = torch.cat([emb_weight, pad], dim=0)
        else:
            emb_weight = emb_weight[:embed_layer.weight.shape[0]]
    embed_layer.weight.copy_(emb_weight)

model.eval()
print("   ✅ Model ready!")

# 6. Inference Function
TASK_LABELS = {
    "A": ["Animal", "Vehicle"],
    "B": ["Natural", "Man-Made"],
    "C": ["Living", "Non-Living"],
}

@torch.no_grad()
def classify(text, task='A'):
    model.switch_task(task)
    tokens = tokenizer(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to(DEVICE)
    logits = model(tokens.input_ids, tokens.attention_mask)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs))
    confidence = float(probs[pred_idx])
    return TASK_LABELS[task][pred_idx], confidence

# 7. Standard Evaluation Test Suite
test_texts = [
    ("A bird", "A", "Animal"),
    ("A vehicle airplane", "A", "Vehicle"),
    ("A car", "A", "Vehicle"),
    ("A cat", "A", "Animal"),
    ("A ship", "A", "Vehicle"),
    ("A dog", "A", "Animal"),
    ("A truck", "A", "Vehicle"),
    ("A horse", "A", "Animal"),
    ("A deer", "A", "Animal"),
    ("A man-made airplane", "B", "Man-Made"),
    ("A bird", "B", "Natural"),
    ("A car", "B", "Man-Made"),
    ("A cat", "B", "Natural"),
    ("A ship", "B", "Man-Made"),
    ("A dog", "B", "Natural"),
    ("A truck", "B", "Man-Made"),
    ("A horse", "B", "Natural"),
    ("A deer", "B", "Natural"),
    ("A non-living airplane", "C", "Non-Living"),
    ("A bird", "C", "Living"),
    ("A car", "C", "Non-Living"),
    ("A cat", "C", "Living"),
    ("A ship", "C", "Non-Living"),
    ("A dog", "C", "Living"),
    ("A truck", "C", "Non-Living"),
    ("A horse", "C", "Living"),
    ("A deer", "C", "Living"),
]

print("\n" + "="*80)
print("📊 RUNNING STANDARD EVALUATION SUITE")
print("="*80)
print(f" {'Task':<6} {'Text':<35} {'Predicted':<12} {'Expected':<12} {'Confidence':<10} {'Status':<6}")
print(f" {'─'*80}")

correct = 0
total = len(test_texts)
evaluation_records = []

for text, task, expected in test_texts:
    label, conf = classify(text, task)
    is_correct = (label == expected)
    status = "✅" if is_correct else "❌"
    if is_correct:
        correct += 1
    evaluation_records.append((is_correct, conf))
    print(f" {task:<6} {text:<35} {label:<12} {expected:<12} {conf*100:.1f}%     {status:<6}")

# 8. Data-Driven Hallucination Score Calculation
def compute_data_driven_hallucination_score(eval_results):
    total_items = len(eval_results)
    errors = 0
    confidence_penalty_sum = 0.0

    for is_correct, conf in eval_results:
        if not is_correct:
            errors += 1
            confidence_penalty_sum += conf
        else:
            if conf < 0.5:
                confidence_penalty_sum += (0.5 - conf)

    error_rate = (errors / total_items) * 100.0
    mean_penalty = (confidence_penalty_sum / total_items) * 100.0
    hallucination_score = (0.7 * error_rate) + (0.3 * mean_penalty)
    return float(hallucination_score)

h_score = compute_data_driven_hallucination_score(evaluation_records)

# 9. Final Audit Report
print("\n" + "="*80)
print("📈 FINAL AUDIT REPORT & DATA-DRIVEN HALLUCINATION SCORE")
print("="*80)
print(f" Total Evaluated: {total}")
print(f" Passed:          {correct}")
print(f" Accuracy:        {correct/total*100:.1f}%")
print(f" Data-Driven Hallucination Score: {h_score:.4f}%")
print("="*80)

🧪 TOPO-2026 FULL EVALUATION & DATA-DRIVEN HALLUCINATION AUDIT
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026

👁️ Loading Vision Model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

✅ Gemma Loaded (Unsloth)

📥 Downloading trained weights from Hugging Face...
   ✅ Checkpoint loaded! Best Task C Accuracy: 100.00%

📥 Loading tokenizer from Hugging Face...

🏗️ Building classifier model...
   ✅ Model ready!

📊 RUNNING STANDARD EVALUATION SUITE
 Task   Text                                Predicted    Expected     Confidence Status
 ────────────────────────────────────────────────────────────────────────────────
 A      A bird                              Animal       Animal       99.9%     ✅     
 A      A vehicle airplane                  Vehicle      Vehicle      100.0%     ✅     
 A      A car                               Vehicle      Vehicle      99.8%     ✅     
 A      A cat                               Animal       Animal       100.0%     ✅     
 A      A ship                              Vehicle      Vehicle      99.7%     ✅     
 A      A dog                               Animal       Animal       100.0%     ✅     
 A      A truck                             

## EXTENTION

In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
import contextlib
import io

print("="*80)
print("🧪 TOPO-2026 STL-10 EXTENDED EVALUATION AUDIT")
print("   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026")
print("="*80)

# 1. Configuration
REPO_ID = "frankmorales2020/gemma-4-e4b-stl10-topo-2026"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 64

# 2. Load Base Model with Unsloth / Transformers Fallback
print("\n👁️ Loading Vision Model...")
vision_model = None
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel
        vision_model, vision_processor = FastVisionModel.from_pretrained(
            "frankmorales2020/gemma-4-e4b-unesco-optimized",
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(vision_model)
    print("✅ Gemma Loaded (Unsloth)")
except Exception as e:
    print(f"⚠️ Unsloth failed: {e}")
    from transformers import AutoModelForCausalLM
    vision_model = AutoModelForCausalLM.from_pretrained(
        "frankmorales2020/gemma-4-e4b-unesco-optimized",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print("✅ Gemma Loaded (Transformers)")

vision_model = vision_model.to(DEVICE)
for param in vision_model.parameters():
    param.requires_grad = False

# 3. Download Checkpoint from HF
print("\n📥 Downloading trained weights from Hugging Face...")
ckpt_path = hf_hub_download(REPO_ID, "topo_trained_parts_gemma_5runs.pt")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"   ✅ Checkpoint loaded! Best Task C Accuracy: {ckpt['best_acc_c']*100:.2f}%")

# 4. Load Tokenizer from HF
print("\n📥 Loading tokenizer from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 5. Build Classifier Model & Load Weights
print("\n🏗️ Building classifier model...")
class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1].float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

model = GemmaTopoClassifier(vision_model, ckpt['hidden_size']).to(DEVICE)
model.classifier_A.load_state_dict(ckpt["classifier_A"])
model.classifier_B.load_state_dict(ckpt["classifier_B"])
model.classifier_C.load_state_dict(ckpt["classifier_C"])

with torch.no_grad():
    emb_weight = ckpt["embed_tokens_weight"].to(DEVICE)
    embed_layer = vision_model.get_input_embeddings()
    if emb_weight.shape != embed_layer.weight.shape:
        if emb_weight.shape[0] < embed_layer.weight.shape[0]:
            pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
            pad = torch.randn(pad_size, emb_weight.shape[1], device=DEVICE)
            emb_weight = torch.cat([emb_weight, pad], dim=0)
        else:
            emb_weight = emb_weight[:embed_layer.weight.shape[0]]
    embed_layer.weight.copy_(emb_weight)

model.eval()
print("   ✅ Model ready!")

# 6. Inference Function
TASK_LABELS = {
    "A": ["Animal", "Vehicle"],
    "B": ["Natural", "Man-Made"],
    "C": ["Living", "Non-Living"],
}

@torch.no_grad()
def classify(text, task='A'):
    model.switch_task(task)
    tokens = tokenizer(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to(DEVICE)
    logits = model(tokens.input_ids, tokens.attention_mask)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs))
    confidence = float(probs[pred_idx])
    return TASK_LABELS[task][pred_idx], confidence

# 7. Extended Test Suite Restricted Strictly to STL-10 Classes
test_texts = [
    # Task A: Animal vs Vehicle (All STL-10 Classes)
    ("A bird", "A", "Animal"),
    ("A cat", "A", "Animal"),
    ("A dog", "A", "Animal"),
    ("A horse", "A", "Animal"),
    ("A deer", "A", "Animal"),
    ("A vehicle airplane", "A", "Vehicle"),
    ("A car", "A", "Vehicle"),
    ("A ship", "A", "Vehicle"),
    ("A truck", "A", "Vehicle"),

    # Task B: Natural vs Man-Made (All STL-10 Classes)
    ("A bird", "B", "Natural"),
    ("A cat", "B", "Natural"),
    ("A dog", "B", "Natural"),
    ("A horse", "B", "Natural"),
    ("A deer", "B", "Natural"),
    ("A man-made airplane", "B", "Man-Made"),
    ("A car", "B", "Man-Made"),
    ("A ship", "B", "Man-Made"),
    ("A truck", "B", "Man-Made"),

    # Task C: Living vs Non-Living (All STL-10 Classes)
    ("A bird", "C", "Living"),
    ("A cat", "C", "Living"),
    ("A dog", "C", "Living"),
    ("A horse", "C", "Living"),
    ("A deer", "C", "Living"),
    ("A non-living airplane", "C", "Non-Living"),
    ("A car", "C", "Non-Living"),
    ("A ship", "C", "Non-Living"),
    ("A truck", "C", "Non-Living"),
]

print("\n" + "="*80)
print(f"📊 RUNNING STRICT STL-10 EXTENDED SUITE ({len(test_texts)} ITEMS)")
print("="*80)
print(f" {'Task':<6} {'Text':<35} {'Predicted':<12} {'Expected':<12} {'Confidence':<10} {'Status':<6}")
print(f" {'─'*80}")

correct = 0
total = len(test_texts)
evaluation_records = []

for text, task, expected in test_texts:
    label, conf = classify(text, task)
    is_correct = (label == expected)
    status = "✅" if is_correct else "❌"
    if is_correct:
        correct += 1
    evaluation_records.append((is_correct, conf))
    print(f" {task:<6} {text:<35} {label:<12} {expected:<12} {conf*100:.1f}%     {status:<6}")

# 8. Data-Driven Hallucination Score Calculation
def compute_data_driven_hallucination_score(eval_results):
    total_items = len(eval_results)
    errors = 0
    confidence_penalty_sum = 0.0

    for is_correct, conf in eval_results:
        if not is_correct:
            errors += 1
            confidence_penalty_sum += conf
        else:
            if conf < 0.5:
                confidence_penalty_sum += (0.5 - conf)

    error_rate = (errors / total_items) * 100.0
    mean_penalty = (confidence_penalty_sum / total_items) * 100.0
    hallucination_score = (0.7 * error_rate) + (0.3 * mean_penalty)
    return float(hallucination_score)

h_score = compute_data_driven_hallucination_score(evaluation_records)

# 9. Final Audit Report
print("\n" + "="*80)
print("📈 FINAL AUDIT REPORT & DATA-DRIVEN HALLUCINATION SCORE")
print("="*80)
print(f" Total Evaluated: {total}")
print(f" Passed:          {correct}")
print(f" Accuracy:        {correct/total*100:.1f}%")
print(f" Data-Driven Hallucination Score: {h_score:.4f}%")
print("="*80)

🧪 TOPO-2026 STL-10 EXTENDED EVALUATION AUDIT
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026

👁️ Loading Vision Model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

✅ Gemma Loaded (Unsloth)

📥 Downloading trained weights from Hugging Face...
   ✅ Checkpoint loaded! Best Task C Accuracy: 100.00%

📥 Loading tokenizer from Hugging Face...

🏗️ Building classifier model...
   ✅ Model ready!

📊 RUNNING STRICT STL-10 EXTENDED SUITE (27 ITEMS)
 Task   Text                                Predicted    Expected     Confidence Status
 ────────────────────────────────────────────────────────────────────────────────
 A      A bird                              Animal       Animal       99.9%     ✅     
 A      A cat                               Animal       Animal       100.0%     ✅     
 A      A dog                               Animal       Animal       100.0%     ✅     
 A      A horse                             Animal       Animal       100.0%     ✅     
 A      A deer                              Animal       Animal       100.0%     ✅     
 A      A vehicle airplane                  Vehicle      Vehicle      100.0%     ✅     
 A      A car               